# HD 189733 b Transit Lab — from MAST FITS to a Physical Transit Depth  
### No Lightkurve workflow | Google Colab version

**Author:** Biswajit Jana  
**Target:** HD 189733 b  
**Method:** TESS FITS light curve → quality masking → local transit-window detrending → phase folding → depth estimate → physical transit model overlay

This notebook is written as a transparent, beginner-friendly but scientifically motivated pipeline.  
It does **not** use `lightkurve`. Instead, it works closer to the actual archive products:

- `astroquery.mast` searches and downloads the public TESS light-curve FITS files from MAST.
- `astropy.io.fits` reads the FITS columns directly.
- The NASA Exoplanet Archive gives the published orbital and physical parameters.
- Each predicted transit window is locally normalised before folding.
- A physical transit model is overplotted using `batman`.
- The measured depth is compared with the expected depth from \( (R_p/R_\star)^2 \).

The purpose is not to claim a final publication-grade fit, but to show how a real transit signal is recovered from public mission data in a way that is inspectable and forkable.

## 1. Install packages

This notebook uses standard astronomy/scientific Python tools rather than `lightkurve`.

`batman-package` is used only for the final physical transit model overlay.

In [ ]:
!pip -q install astroquery astropy batman-package scipy pandas numpy matplotlib

## 2. Imports and project settings

You can change `TARGET_NAME` and `PLANET_NAME` later to test another known transiting planet.  
For this first version, HD 189733 b is useful because it is a deep hot-Jupiter transit.

In [ ]:
from pathlib import Path
from urllib.parse import quote
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from astroquery.mast import Observations
from astropy.io import fits
from astropy.stats import sigma_clip
from astropy.timeseries import BoxLeastSquares
import astropy.units as u

from scipy.optimize import curve_fit

try:
    import batman
    BATMAN_AVAILABLE = True
except Exception as exc:
    BATMAN_AVAILABLE = False
    print("batman import failed; physical model overlay will be skipped:", exc)

# -----------------------
# User-facing settings
# -----------------------
TARGET_NAME = "HD 189733"
PLANET_NAME = "HD 189733 b"

# To keep Colab fast, start with one TESS light-curve file.
# Increase to 2 or 3 if you want more transits after the first run works.
MAX_LIGHTCURVE_FILES = 1

# Local transit extraction settings
WINDOW_DAYS = 0.30          # extract +/- 0.30 d around each predicted transit
BIN_MINUTES = 10            # binned folded curve resolution
QUALITY_MODE = "strict"     # "strict" uses QUALITY == 0; "loose" keeps all finite fluxes

OUTDIR = Path("outputs")
DATADIR = Path("data")
OUTDIR.mkdir(exist_ok=True)
DATADIR.mkdir(exist_ok=True)

plt.rcParams.update({
    "figure.figsize": (10, 5),
    "axes.grid": True,
    "font.size": 11,
})

## 3. Download planet parameters from NASA Exoplanet Archive

The archive values are used as the starting ephemeris:

- orbital period
- reference transit midpoint
- transit duration
- planet/star radius ratio where available
- \(a/R_\star\), inclination and other parameters if available

Using the known ephemeris is important here because HD 189733 is an active star; blindly searching with a periodogram can sometimes lock onto stellar/instrumental variability instead of the real transit.

In [ ]:
def query_exoplanet_archive(planet_name: str) -> pd.DataFrame:
    """Query the NASA Exoplanet Archive PSCompPars table using TAP."""
    base = "https://exoplanetarchive.ipac.caltech.edu/TAP/sync"
    sql = f"SELECT * FROM pscomppars WHERE pl_name = '{planet_name}'"
    url = f"{base}?query={quote(sql)}&format=csv"
    return pd.read_csv(url)


def get_value(row: pd.Series, key: str, default=np.nan):
    """Safe getter for archive columns."""
    if key in row.index and pd.notna(row[key]):
        return row[key]
    return default


props_df = query_exoplanet_archive(PLANET_NAME)

if len(props_df) == 0:
    raise RuntimeError(f"No NASA Exoplanet Archive entry found for {PLANET_NAME}")

props = props_df.iloc[0]

# Core ephemeris
PERIOD_DAYS = float(get_value(props, "pl_orbper"))
T0_BJD = float(get_value(props, "pl_tranmid"))
DURATION_HOURS = float(get_value(props, "pl_trandur"))

# Useful physical/model parameters; fallbacks are literature-style approximate values for HD 189733 b.
RP_RS_ARCHIVE = float(get_value(props, "pl_ratror", np.nan))
A_RS = float(get_value(props, "pl_ratdor", 8.9))
INC_DEG = float(get_value(props, "pl_orbincl", 85.7))
ECC = float(get_value(props, "pl_orbeccen", 0.0))
OMEGA_DEG = float(get_value(props, "pl_orblper", 90.0))

# Sometimes omega is NaN for circular orbits; batman needs a number.
if not np.isfinite(OMEGA_DEG):
    OMEGA_DEG = 90.0
if not np.isfinite(ECC):
    ECC = 0.0

selected = {
    "Planet": PLANET_NAME,
    "Host star": TARGET_NAME,
    "Period [days]": PERIOD_DAYS,
    "Transit midpoint T0 [BJD]": T0_BJD,
    "Transit duration [hours]": DURATION_HOURS,
    "Rp/Rs from archive": RP_RS_ARCHIVE,
    "Expected depth from Rp/Rs [%]": (RP_RS_ARCHIVE**2 * 100) if np.isfinite(RP_RS_ARCHIVE) else np.nan,
    "a/Rs": A_RS,
    "Inclination [deg]": INC_DEG,
    "Eccentricity": ECC,
    "Stellar radius [Rsun]": get_value(props, "st_rad"),
    "Planet radius [Rjup]": get_value(props, "pl_radj"),
    "Distance [pc]": get_value(props, "sy_dist"),
    "Discovery method": get_value(props, "discoverymethod", ""),
}

params_table = pd.DataFrame.from_dict(selected, orient="index", columns=["value"])
display(params_table)

props_df.to_csv(OUTDIR / "01_nasa_exoplanet_archive_pscomppars.csv", index=False)
params_table.to_csv(OUTDIR / "02_selected_target_parameters.csv")

## 4. Search MAST for public TESS light-curve FITS files

This section searches MAST directly and filters for extracted TESS light curves (`*_lc.fits`).  
Only the first file is downloaded by default to keep the notebook fast and reliable for community users.

You can increase `MAX_LIGHTCURVE_FILES` after confirming the first run works.

In [ ]:
def search_tess_lightcurve_products(target_name: str):
    """Search MAST around the target and return likely TESS light-curve FITS products."""
    print(f"Searching MAST for: {target_name}")

    obs = Observations.query_object(
        target_name,
        radius="0.02 deg"
    )

    if len(obs) == 0:
        raise RuntimeError("No MAST observations found around this target.")

    # Keep only TESS observations
    obs = obs[obs["obs_collection"] == "TESS"]

    if len(obs) == 0:
        raise RuntimeError("No TESS observations found around this target.")

    products = Observations.get_product_list(obs)
    pdf = products.to_pandas()

    # Typical SPOC light-curve files end with _lc.fits
    name = pdf["productFilename"].astype(str)
    mask_lc = name.str.contains("_lc.fits", case=False, regex=False)

    if "productType" in pdf.columns:
        mask_lc &= pdf["productType"].astype(str).str.upper().eq("SCIENCE")

    lc_pdf = pdf[mask_lc].copy()

    if len(lc_pdf) == 0:
        raise RuntimeError("No *_lc.fits TESS light-curve products found.")

    # Prefer SPOC-like filenames beginning with tess, but keep others if needed.
    lc_pdf["is_spoc_like"] = lc_pdf["productFilename"].astype(str).str.startswith("tess")

    # Sort in a stable way. Columns vary slightly across product tables, so keep this defensive.
    sort_cols = ["is_spoc_like"]
    ascending = [False]
    for c in ["sequence_number", "t_min"]:
        if c in lc_pdf.columns:
            sort_cols.append(c)
            ascending.append(True)

    lc_pdf = lc_pdf.sort_values(sort_cols, ascending=ascending).reset_index(drop=True)
    return lc_pdf, products


lc_products_df, all_products_table = search_tess_lightcurve_products(TARGET_NAME)

print(f"Found {len(lc_products_df)} likely TESS light-curve FITS products.")
show_cols = [c for c in ["productFilename", "size", "is_spoc_like"] if c in lc_products_df.columns]
display(lc_products_df[show_cols].head(10))

## 5. Download selected FITS files

This uses `astroquery.mast.Observations.download_products` and saves files under `data/`.

In [ ]:
def download_selected_products(products_table, lc_products_df, max_files=1):
    """Download selected light-curve products and return local FITS paths."""
    selected_indices = lc_products_df.index[:max_files].tolist()

    # Convert selected product rows back into an Astropy table by matching filenames.
    selected_names = set(lc_products_df.loc[selected_indices, "productFilename"].astype(str))
    selected_mask = [str(x) in selected_names for x in products_table["productFilename"]]
    selected_products = products_table[selected_mask]

    print("Downloading these products:")
    for name in selected_products["productFilename"]:
        print(" -", name)

    manifest = Observations.download_products(
        selected_products,
        download_dir=str(DATADIR)
    )

    paths = []
    for p in manifest["Local Path"]:
        p = Path(str(p))
        if p.exists() and p.name.lower().endswith(".fits"):
            paths.append(p)

    if len(paths) == 0:
        raise RuntimeError("Download finished but no FITS files were found locally.")

    return paths


fits_paths = download_selected_products(
    all_products_table,
    lc_products_df,
    max_files=MAX_LIGHTCURVE_FILES
)

print("Local FITS files:")
for p in fits_paths:
    print(" -", p)

## 6. Read TESS FITS columns directly

Most TESS light-curve files include columns such as:

- `TIME`
- `SAP_FLUX`
- `PDCSAP_FLUX`
- `QUALITY`

For transit work, `PDCSAP_FLUX` is normally useful because it is systematics-corrected by the pipeline.  
We still inspect and normalise it ourselves.

In [ ]:
def read_tess_lc_fits(path: Path, quality_mode="strict") -> pd.DataFrame:
    """Read a TESS light-curve FITS file without Lightkurve."""
    with fits.open(path) as hdul:
        primary_header = hdul[0].header
        data_header = hdul[1].header
        data = hdul[1].data
        names = list(data.names)

        flux_col = "PDCSAP_FLUX" if "PDCSAP_FLUX" in names else "SAP_FLUX"
        err_col = f"{flux_col}_ERR" if f"{flux_col}_ERR" in names else None

        time = np.asarray(data["TIME"], dtype=float)
        flux = np.asarray(data[flux_col], dtype=float)
        quality = np.asarray(data["QUALITY"], dtype=int) if "QUALITY" in names else np.zeros_like(time, dtype=int)

        if err_col is not None:
            flux_err = np.asarray(data[err_col], dtype=float)
        else:
            flux_err = np.full_like(flux, np.nan)

        good = np.isfinite(time) & np.isfinite(flux)
        if quality_mode == "strict":
            good &= quality == 0

        df = pd.DataFrame({
            "time": time[good],
            "flux": flux[good],
            "flux_err": flux_err[good],
            "quality": quality[good],
            "source_file": path.name,
            "flux_column": flux_col,
            "sector": primary_header.get("SECTOR", data_header.get("SECTOR", np.nan)),
            "camera": primary_header.get("CAMERA", data_header.get("CAMERA", np.nan)),
            "ccd": primary_header.get("CCD", data_header.get("CCD", np.nan)),
        })

        return df


frames = []
for path in fits_paths:
    frames.append(read_tess_lc_fits(path, quality_mode=QUALITY_MODE))

lc_df = pd.concat(frames, ignore_index=True).sort_values("time").reset_index(drop=True)

if len(lc_df) < 50 and QUALITY_MODE == "strict":
    warnings.warn("Strict quality mask left very few points. Re-reading with loose quality mask.")
    frames = [read_tess_lc_fits(p, quality_mode="loose") for p in fits_paths]
    lc_df = pd.concat(frames, ignore_index=True).sort_values("time").reset_index(drop=True)

# Normalise flux for plotting and analysis
lc_df["flux_norm"] = lc_df["flux"] / np.nanmedian(lc_df["flux"])

if np.isfinite(lc_df["flux_err"]).sum() > 0:
    lc_df["flux_err_norm"] = lc_df["flux_err"] / np.nanmedian(lc_df["flux"])
else:
    lc_df["flux_err_norm"] = np.nan

print(f"Loaded {len(lc_df)} quality-masked data points.")
display(lc_df.head())

lc_df.to_csv(OUTDIR / "03_quality_masked_tess_lightcurve.csv", index=False)

## 7. Plot the raw quality-masked TESS light curve

This is not the final transit plot.  
The raw time series can contain stellar rotation, spots, instrumental trends and sector-level systematics.

For HD 189733, stellar activity is part of the real astrophysical context, so the raw curve is expected to be imperfect.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))

ax.scatter(
    lc_df["time"],
    lc_df["flux_norm"],
    s=4,
    alpha=0.45,
    linewidths=0,
)

ax.set_title(f"{TARGET_NAME}: raw quality-masked TESS light curve from FITS")
ax.set_xlabel("Time [BTJD-like TESS days]")
ax.set_ylabel("Normalised flux")

lo, hi = np.nanpercentile(lc_df["flux_norm"], [1, 99])
pad = 0.002
ax.set_ylim(lo - pad, hi + pad)

fig.tight_layout()
fig.savefig(OUTDIR / "04_raw_quality_masked_tess_lightcurve.png", dpi=250)
plt.show()

## 8. Convert archive transit epoch into the TESS time system

TESS light-curve `TIME` values are normally stored as a reduced time system close to:

\[
\mathrm{BTJD} = \mathrm{BJD} - 2457000
\]

The archive transit midpoint is usually given as a full BJD.  
This function shifts it into the same time range as the downloaded TESS light curve.

In [ ]:
def convert_bjd_to_observed_time(t0_bjd, observed_time, period_days):
    """Convert full BJD transit epoch into the light-curve time system and shift near the data."""
    median_time = np.nanmedian(observed_time)

    # Try the common mission time offsets.
    candidates = {
        "BJD": t0_bjd,
        "BTJD = BJD - 2457000": t0_bjd - 2457000.0,
        "BKJD = BJD - 2454833": t0_bjd - 2454833.0,
    }

    base_name, base_t0 = min(
        candidates.items(),
        key=lambda item: abs(item[1] - median_time)
    )

    # Shift by integer periods so that the reference transit lies close to observed data.
    n = np.round((median_time - base_t0) / period_days)
    shifted_t0 = base_t0 + n * period_days

    return shifted_t0, base_name, int(n)


T0_OBS, TIME_SYSTEM_USED, N_SHIFT = convert_bjd_to_observed_time(
    T0_BJD,
    lc_df["time"].to_numpy(),
    PERIOD_DAYS
)

print(f"Archive T0 [BJD]          : {T0_BJD}")
print(f"Converted using           : {TIME_SYSTEM_USED}")
print(f"Integer-period shift n    : {N_SHIFT}")
print(f"T0 shifted into TESS data : {T0_OBS:.8f}")

## 9. Predict all transit centres in the downloaded TESS data

This creates a list of expected transit midpoints inside the observed time span.  
These midpoints define the windows we will extract and locally normalise.

In [ ]:
def predicted_transit_times(time, t0_near, period_days):
    """Return predicted transit midpoints covering the observed time range."""
    tmin, tmax = np.nanmin(time), np.nanmax(time)

    n_start = int(np.floor((tmin - t0_near) / period_days)) - 2
    n_end = int(np.ceil((tmax - t0_near) / period_days)) + 2

    centres = t0_near + np.arange(n_start, n_end + 1) * period_days
    centres = centres[(centres > tmin) & (centres < tmax)]

    return centres


transit_centres = predicted_transit_times(
    lc_df["time"].to_numpy(),
    T0_OBS,
    PERIOD_DAYS
)

print(f"Number of predicted transits in downloaded data: {len(transit_centres)}")
print(transit_centres[:10])

## 10. Visual sanity check: predicted transits on the raw time series

The vertical lines mark where the transit should occur from the published ephemeris.  
This is a useful sanity check before any folding.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))

ax.scatter(
    lc_df["time"],
    lc_df["flux_norm"],
    s=4,
    alpha=0.40,
    linewidths=0,
    label="Quality-masked flux"
)

for t0 in transit_centres:
    ax.axvline(t0, color="tab:red", alpha=0.25, linewidth=1)

ax.set_title(f"{PLANET_NAME}: predicted transit centres on raw TESS data")
ax.set_xlabel("Time [BTJD-like TESS days]")
ax.set_ylabel("Normalised flux")

lo, hi = np.nanpercentile(lc_df["flux_norm"], [1, 99])
ax.set_ylim(lo - 0.002, hi + 0.002)
ax.legend(loc="best")

fig.tight_layout()
fig.savefig(OUTDIR / "05_predicted_transits_on_raw_flux.png", dpi=250)
plt.show()

## 11. Local transit-window normalisation

This is the key step.

For each predicted transit:

1. extract a window around the transit,
2. exclude the transit itself,
3. fit a low-order polynomial baseline to the out-of-transit points,
4. divide the local flux by that baseline,
5. combine all locally normalised transits.

This is more transparent than hiding everything inside a single library call.

In [ ]:
def robust_polyfit_baseline(x, y, degree=1, maxiters=5, sigma=4):
    """Fit a robust polynomial baseline using sigma clipping."""
    mask = np.isfinite(x) & np.isfinite(y)

    if mask.sum() < degree + 3:
        return None

    x_fit = x[mask]
    y_fit = y[mask]

    for _ in range(maxiters):
        coeff = np.polyfit(x_fit, y_fit, deg=degree)
        model = np.polyval(coeff, x_fit)
        resid = y_fit - model
        clipped = sigma_clip(resid, sigma=sigma, maxiters=1)
        keep = ~clipped.mask

        if keep.sum() < degree + 3:
            break

        x_fit = x_fit[keep]
        y_fit = y_fit[keep]

    return np.polyfit(x_fit, y_fit, deg=degree)


def extract_and_normalise_transits(
    lc_df,
    transit_centres,
    duration_hours,
    window_days=0.30,
    poly_degree=1
):
    """Extract predicted transit windows and locally normalise each window."""

    time = lc_df["time"].to_numpy(dtype=float)
    flux = lc_df["flux_norm"].to_numpy(dtype=float)
    flux_err = lc_df["flux_err_norm"].to_numpy(dtype=float)

    duration_days = duration_hours / 24.0

    chunks = []

    for i, centre in enumerate(transit_centres, start=1):
        in_window = np.abs(time - centre) <= window_days

        if in_window.sum() < 30:
            continue

        local_time = time[in_window] - centre
        local_flux = flux[in_window]
        local_err = flux_err[in_window]

        # Out-of-transit region used for baseline fitting.
        # We exclude a region slightly wider than the nominal transit duration.
        oot = np.abs(local_time) > 0.75 * duration_days

        if oot.sum() < 20:
            continue

        coeff = robust_polyfit_baseline(
            local_time[oot],
            local_flux[oot],
            degree=poly_degree,
            maxiters=5,
            sigma=4
        )

        if coeff is None:
            continue

        baseline = np.polyval(coeff, local_time)

        # Avoid division by bad baselines.
        ok = np.isfinite(baseline) & (baseline > 0)

        source_vals = lc_df.loc[in_window, "source_file"].to_numpy()

        chunk = pd.DataFrame({
            "transit_number": i,
            "transit_centre_time": centre,
            "phase_days": local_time[ok],
            "phase_hours": local_time[ok] * 24.0,
            "flux_local_norm": local_flux[ok] / baseline[ok],
            "flux_err_local_norm": local_err[ok] / baseline[ok] if np.isfinite(local_err).sum() > 0 else np.nan,
            "baseline": baseline[ok],
            "source_file": source_vals[ok],
        })

        chunks.append(chunk)

    if len(chunks) == 0:
        raise RuntimeError("No transit windows could be normalised. Try increasing WINDOW_DAYS or using loose quality mode.")

    out = pd.concat(chunks, ignore_index=True)

    # Remove extreme outliers after local normalisation.
    med = np.nanmedian(out["flux_local_norm"])
    std = np.nanstd(out["flux_local_norm"])
    keep = np.abs(out["flux_local_norm"] - med) < 5 * std

    return out[keep].reset_index(drop=True)


fold_df = extract_and_normalise_transits(
    lc_df,
    transit_centres,
    DURATION_HOURS,
    window_days=WINDOW_DAYS,
    poly_degree=1
)

print(f"Locally normalised points: {len(fold_df)}")
print(f"Transits used: {fold_df['transit_number'].nunique()}")

fold_df.to_csv(OUTDIR / "06_locally_normalised_transit_points.csv", index=False)
display(fold_df.head())

## 12. Bin the folded transit and estimate depth

Depth is estimated from median in-transit and out-of-transit flux:

\[
\delta = 1 - \frac{F_\mathrm{in}}{F_\mathrm{out}}
\]

This is a simple diagnostic depth estimate, not a full posterior fit.

In [ ]:
def bin_folded_curve(fold_df, bin_minutes=10):
    """Median-bin a folded light curve in phase hours."""
    bin_hr = bin_minutes / 60.0
    phase = fold_df["phase_hours"].to_numpy(dtype=float)
    flux = fold_df["flux_local_norm"].to_numpy(dtype=float)

    bins = np.arange(-WINDOW_DAYS * 24, WINDOW_DAYS * 24 + bin_hr, bin_hr)
    centres = 0.5 * (bins[:-1] + bins[1:])

    rows = []
    for lo, hi, c in zip(bins[:-1], bins[1:], centres):
        m = (phase >= lo) & (phase < hi)
        if m.sum() >= 3:
            rows.append({
                "phase_hours": c,
                "flux_median": np.nanmedian(flux[m]),
                "flux_mean": np.nanmean(flux[m]),
                "flux_std": np.nanstd(flux[m]),
                "n_points": int(m.sum()),
                "flux_sem": np.nanstd(flux[m]) / np.sqrt(m.sum())
            })

    return pd.DataFrame(rows)


def estimate_depth(fold_df, duration_hours):
    """Estimate transit depth from in-transit and out-of-transit medians."""
    phase = fold_df["phase_hours"].to_numpy(dtype=float)
    flux = fold_df["flux_local_norm"].to_numpy(dtype=float)

    in_transit = np.abs(phase) <= duration_hours / 2.0
    oot = (np.abs(phase) >= 1.5 * duration_hours) & (np.abs(phase) <= WINDOW_DAYS * 24 * 0.85)

    f_in = np.nanmedian(flux[in_transit])
    f_out = np.nanmedian(flux[oot])

    depth = 1.0 - (f_in / f_out)

    return {
        "f_in": f_in,
        "f_out": f_out,
        "depth_fraction": depth,
        "depth_percent": 100 * depth,
        "depth_ppm": 1e6 * depth,
        "n_in": int(in_transit.sum()),
        "n_out": int(oot.sum()),
    }


def bootstrap_depth(fold_df, duration_hours, n_boot=1000, seed=42):
    """Bootstrap uncertainty for the median depth estimate."""
    rng = np.random.default_rng(seed)
    n = len(fold_df)
    depths = []

    for _ in range(n_boot):
        sample = fold_df.iloc[rng.integers(0, n, n)].reset_index(drop=True)
        d = estimate_depth(sample, duration_hours)["depth_percent"]
        if np.isfinite(d):
            depths.append(d)

    depths = np.array(depths)

    return {
        "depth_percent_median": np.nanmedian(depths),
        "depth_percent_std": np.nanstd(depths),
        "depth_percent_p16": np.nanpercentile(depths, 16),
        "depth_percent_p84": np.nanpercentile(depths, 84),
    }


binned_df = bin_folded_curve(fold_df, bin_minutes=BIN_MINUTES)
depth_info = estimate_depth(fold_df, DURATION_HOURS)
depth_unc = bootstrap_depth(fold_df, DURATION_HOURS, n_boot=500)

expected_depth_percent = (RP_RS_ARCHIVE**2 * 100) if np.isfinite(RP_RS_ARCHIVE) else np.nan

print(f"Measured depth       : {depth_info['depth_percent']:.3f}%  ({depth_info['depth_ppm']:.0f} ppm)")
print(f"Bootstrap 16-84%     : {depth_unc['depth_percent_p16']:.3f}% to {depth_unc['depth_percent_p84']:.3f}%")
print(f"Expected (Rp/Rs)^2   : {expected_depth_percent:.3f}%")

binned_df.to_csv(OUTDIR / "07_binned_folded_transit.csv", index=False)
display(binned_df.head())

## 13. Fit a simple physical transit model using `batman`

This is a compact model overlay, not a full MCMC analysis.

We hold most orbital geometry close to archive/literature values and fit mainly:

- \(R_p/R_\star\)
- small flux offset

For a more advanced repo, this can later become a full Bayesian fit with priors.

In [ ]:
def batman_model_flux(phase_hours, rp_rs, flux_offset):
    """Compute batman transit model for folded phase times."""
    if not BATMAN_AVAILABLE:
        return np.ones_like(phase_hours) * np.nan

    phase_days = np.asarray(phase_hours, dtype=float) / 24.0

    params = batman.TransitParams()
    params.t0 = 0.0
    params.per = PERIOD_DAYS
    params.rp = rp_rs
    params.a = A_RS if np.isfinite(A_RS) and A_RS > 1 else 8.9
    params.inc = INC_DEG if np.isfinite(INC_DEG) else 85.7
    params.ecc = ECC if np.isfinite(ECC) else 0.0
    params.w = OMEGA_DEG if np.isfinite(OMEGA_DEG) else 90.0

    # Approximate quadratic limb darkening for a K star in a broad optical/TESS-like band.
    # This is intentionally simple for the tutorial.
    params.u = [0.35, 0.20]
    params.limb_dark = "quadratic"

    m = batman.TransitModel(params, phase_days)
    return m.light_curve(params) + flux_offset


model_result = {
    "fit_success": False,
    "rp_rs_fit": np.nan,
    "rp_rs_fit_err": np.nan,
    "model_depth_percent": np.nan,
    "flux_offset": np.nan,
}

if BATMAN_AVAILABLE:
    fit_data = binned_df.dropna(subset=["phase_hours", "flux_median"]).copy()

    # Restrict the fit to the transit neighbourhood.
    fit_data = fit_data[np.abs(fit_data["phase_hours"]) <= 5.0]

    xfit = fit_data["phase_hours"].to_numpy(dtype=float)
    yfit = fit_data["flux_median"].to_numpy(dtype=float)
    sigma = fit_data["flux_sem"].to_numpy(dtype=float)

    good_sigma = np.isfinite(sigma) & (sigma > 0)
    if good_sigma.sum() > 0:
        sigma[~good_sigma] = np.nanmedian(sigma[good_sigma])
    else:
        sigma[:] = 0.001

    initial_rp = np.sqrt(max(depth_info["depth_fraction"], 1e-6))
    if np.isfinite(RP_RS_ARCHIVE):
        initial_rp = RP_RS_ARCHIVE

    try:
        popt, pcov = curve_fit(
            batman_model_flux,
            xfit,
            yfit,
            p0=[initial_rp, 0.0],
            sigma=sigma,
            absolute_sigma=False,
            bounds=([0.05, -0.02], [0.30, 0.02]),
            maxfev=10000
        )

        rp_fit, flux_offset_fit = popt
        perr = np.sqrt(np.diag(pcov))

        model_result.update({
            "fit_success": True,
            "rp_rs_fit": rp_fit,
            "rp_rs_fit_err": perr[0],
            "model_depth_percent": rp_fit**2 * 100,
            "flux_offset": flux_offset_fit,
        })

        print(f"batman fit Rp/Rs       : {rp_fit:.5f} ± {perr[0]:.5f}")
        print(f"batman model depth     : {rp_fit**2 * 100:.3f}%")
        print(f"archive Rp/Rs          : {RP_RS_ARCHIVE:.5f}" if np.isfinite(RP_RS_ARCHIVE) else "archive Rp/Rs unavailable")

    except Exception as exc:
        print("batman fit failed; plot will show archive-style model if possible:", exc)
else:
    print("batman not available, skipping physical model fit.")

## 14. Final science plot: folded transit + binned curve + physical model

This is the main plot for the GitHub README or LinkedIn/community post.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

# Scatter all locally normalised points
ax.scatter(
    fold_df["phase_hours"],
    fold_df["flux_local_norm"],
    s=5,
    alpha=0.18,
    linewidths=0,
    label="Locally normalised TESS points"
)

# Binned curve
ax.errorbar(
    binned_df["phase_hours"],
    binned_df["flux_median"],
    yerr=binned_df["flux_sem"],
    fmt="o-",
    markersize=4,
    linewidth=1.6,
    capsize=2,
    label=f"{BIN_MINUTES}-min median bins"
)

# Model overlay
x_model = np.linspace(-5, 5, 1000)
if BATMAN_AVAILABLE:
    if model_result["fit_success"]:
        y_model = batman_model_flux(x_model, model_result["rp_rs_fit"], model_result["flux_offset"])
        model_label = f"batman model fit: Rp/Rs={model_result['rp_rs_fit']:.4f}"
    elif np.isfinite(RP_RS_ARCHIVE):
        y_model = batman_model_flux(x_model, RP_RS_ARCHIVE, 0.0)
        model_label = f"batman model using archive Rp/Rs={RP_RS_ARCHIVE:.4f}"
    else:
        y_model = None
        model_label = None

    if y_model is not None:
        ax.plot(x_model, y_model, linewidth=2.5, label=model_label)

ax.axvline(0, linestyle="--", linewidth=1, alpha=0.6)
ax.axhline(1, linestyle=":", linewidth=1, alpha=0.6)

ax.set_xlim(-5, 5)

# Plot limits chosen for a deep hot-Jupiter transit
y_low = min(0.965, np.nanpercentile(fold_df["flux_local_norm"], 1) - 0.003)
y_high = max(1.010, np.nanpercentile(fold_df["flux_local_norm"], 99) + 0.003)
ax.set_ylim(y_low, y_high)

title = (
    f"{PLANET_NAME}: TESS transit recovered directly from FITS files\n"
    f"Measured depth = {depth_info['depth_percent']:.3f}% "
    f"({depth_info['depth_ppm']:.0f} ppm); "
    f"Expected ≈ {expected_depth_percent:.3f}%"
)
ax.set_title(title)
ax.set_xlabel("Time from mid-transit [hours]")
ax.set_ylabel("Locally normalised flux")
ax.legend(loc="best", frameon=True)

fig.tight_layout()
fig.savefig(OUTDIR / "08_HD189733b_final_transit_model_plot.png", dpi=300)
plt.show()

## 15. Residual plot

A residual plot helps show what remains after subtracting the simple physical model.  
For HD 189733 b, residuals can include starspots, imperfect detrending, limb-darkening mismatch, and normal photometric scatter.

In [ ]:
if BATMAN_AVAILABLE:
    fig, ax = plt.subplots(figsize=(10, 4.5))

    resid_df = binned_df.dropna(subset=["phase_hours", "flux_median"]).copy()

    if model_result["fit_success"]:
        model_for_bins = batman_model_flux(
            resid_df["phase_hours"].to_numpy(),
            model_result["rp_rs_fit"],
            model_result["flux_offset"]
        )
    elif np.isfinite(RP_RS_ARCHIVE):
        model_for_bins = batman_model_flux(
            resid_df["phase_hours"].to_numpy(),
            RP_RS_ARCHIVE,
            0.0
        )
    else:
        model_for_bins = np.ones(len(resid_df))

    residual_ppm = (resid_df["flux_median"].to_numpy() - model_for_bins) * 1e6

    ax.axhline(0, linestyle="--", linewidth=1, alpha=0.7)
    ax.errorbar(
        resid_df["phase_hours"],
        residual_ppm,
        yerr=resid_df["flux_sem"] * 1e6,
        fmt="o",
        markersize=4,
        capsize=2,
    )

    ax.set_xlim(-5, 5)
    ax.set_title(f"{PLANET_NAME}: residuals after simple transit model")
    ax.set_xlabel("Time from mid-transit [hours]")
    ax.set_ylabel("Residual [ppm]")

    fig.tight_layout()
    fig.savefig(OUTDIR / "09_transit_model_residuals.png", dpi=300)
    plt.show()
else:
    print("Residual plot skipped because batman is unavailable.")

## 16. Optional sanity check: narrow BLS search around the known period

Box Least Squares is a useful transit-search diagnostic, but for active stars it should not replace the published ephemeris blindly.  
Here it is used as a sanity check around the known period, not as the main folding method.

In [ ]:
try:
    time = lc_df["time"].to_numpy(dtype=float)
    flux = lc_df["flux_norm"].to_numpy(dtype=float)

    # Detrend gently by subtracting a rolling median-like long trend using pandas.
    # This is only for BLS visualisation, not the final transit extraction.
    temp = pd.DataFrame({"time": time, "flux": flux}).sort_values("time")
    temp["trend"] = temp["flux"].rolling(window=301, center=True, min_periods=20).median()
    temp["trend"] = temp["trend"].interpolate().bfill().ffill()
    temp["flat"] = temp["flux"] / temp["trend"]

    good = np.isfinite(temp["time"]) & np.isfinite(temp["flat"])

    t_bls = temp.loc[good, "time"].to_numpy()
    y_bls = temp.loc[good, "flat"].to_numpy()

    period_grid = np.linspace(PERIOD_DAYS * 0.98, PERIOD_DAYS * 1.02, 500)
    duration_grid = np.linspace((DURATION_HOURS / 24) * 0.6, (DURATION_HOURS / 24) * 1.5, 12)

    bls = BoxLeastSquares(t_bls * u.day, y_bls)
    bls_power = bls.power(period_grid * u.day, duration_grid * u.day, objective="snr")

    best = np.nanargmax(bls_power.power)
    bls_best_period = bls_power.period[best].value
    bls_best_t0 = bls_power.transit_time[best].value
    bls_best_duration = bls_power.duration[best].value * 24

    print(f"Archive period      : {PERIOD_DAYS:.8f} d")
    print(f"BLS best period     : {bls_best_period:.8f} d")
    print(f"BLS best duration   : {bls_best_duration:.3f} h")

    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.plot(bls_power.period.value, bls_power.power, linewidth=1.5)
    ax.axvline(PERIOD_DAYS, linestyle="--", linewidth=1, label="Archive period")
    ax.axvline(bls_best_period, linestyle=":", linewidth=1.5, label="BLS best")
    ax.set_title(f"{PLANET_NAME}: narrow BLS sanity check")
    ax.set_xlabel("Period [days]")
    ax.set_ylabel("BLS power / SNR objective")
    ax.legend()

    fig.tight_layout()
    fig.savefig(OUTDIR / "10_narrow_bls_sanity_check.png", dpi=250)
    plt.show()

except Exception as exc:
    print("BLS sanity check skipped:", exc)

## 17. Save final summary

This creates a CSV table with the measured depth, expected depth, model-fit parameters and key run settings.

In [ ]:
summary = {
    "target_name": TARGET_NAME,
    "planet_name": PLANET_NAME,
    "n_lightcurve_files": len(fits_paths),
    "n_points_quality_masked": len(lc_df),
    "n_predicted_transits": len(transit_centres),
    "n_transits_used": fold_df["transit_number"].nunique(),
    "period_days_archive": PERIOD_DAYS,
    "t0_bjd_archive": T0_BJD,
    "duration_hours_archive": DURATION_HOURS,
    "rp_rs_archive": RP_RS_ARCHIVE,
    "expected_depth_percent_from_archive_rp_rs": expected_depth_percent,
    "measured_depth_percent": depth_info["depth_percent"],
    "measured_depth_ppm": depth_info["depth_ppm"],
    "bootstrap_depth_percent_p16": depth_unc["depth_percent_p16"],
    "bootstrap_depth_percent_p84": depth_unc["depth_percent_p84"],
    "batman_fit_success": model_result["fit_success"],
    "batman_rp_rs_fit": model_result["rp_rs_fit"],
    "batman_rp_rs_fit_err": model_result["rp_rs_fit_err"],
    "batman_model_depth_percent": model_result["model_depth_percent"],
    "quality_mode": QUALITY_MODE,
    "window_days": WINDOW_DAYS,
    "bin_minutes": BIN_MINUTES,
}

summary_df = pd.DataFrame([summary])
summary_df.to_csv(OUTDIR / "11_final_science_summary.csv", index=False)

display(summary_df.T.rename(columns={0: "value"}))
print("Saved all outputs in:", OUTDIR.resolve())

## 18. What this notebook demonstrates

This notebook is meant to show the full logic of a transit recovery pipeline:

1. get public mission data from MAST,
2. read the FITS file directly,
3. apply quality flags,
4. use a known ephemeris from NASA Exoplanet Archive,
5. isolate individual predicted transits,
6. locally remove baseline trends,
7. fold and bin the transit,
8. estimate the transit depth,
9. compare against \( (R_p/R_\star)^2 \),
10. overlay a physical transit model.

### Suggested next upgrades

- Add a target dropdown for several hot Jupiters.
- Add automatic sector comparison.
- Add a real limb-darkening lookup instead of approximate coefficients.
- Fit more parameters using MCMC.
- Add a simple web dashboard version for citizen scientists.
- Add uploaded CSV support for ground-based Exoplanet Watch observations.